# Airbnb SBERT Embeddings

Run in Colab with a GPU. This notebook reads compact text-input parquets from `data/processed` and writes joinable embedding parquets keyed by `city`, `snapshot`, and `listing_id`.


In [ ]:
!pip -q install sentence-transformers pyarrow

In [ ]:
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

OUT_DIR = Path("/content/drive/MyDrive/comp90051/data/processed")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 512
RUN_SPLITS = {"train", "test", "generalization"}

In [ ]:
def embed_texts(model, texts, prefix):
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return pd.DataFrame(
        vectors,
        columns=[f"{prefix}_sbert_{i:03d}" for i in range(vectors.shape[1])],
    )

In [ ]:
model = SentenceTransformer(MODEL_NAME, device="cuda")

In [ ]:
for split in sorted(RUN_SPLITS):
    input_path = OUT_DIR / f"airbnb_sbert_input_{split}.parquet"
    frame = pd.read_parquet(input_path)
    frame["host_about_text"] = frame["host_about_text"].fillna("")
    frame["review_text"] = frame["review_text"].fillna("")

    print(f"Encoding {split}: {len(frame):,} rows")
    host_emb = embed_texts(model, frame["host_about_text"].tolist(), "host_about")
    review_emb = embed_texts(model, frame["review_text"].tolist(), "reviews")
    keys = frame[["city", "snapshot", "listing_id"]].reset_index(drop=True)
    embeddings = pd.concat([keys, host_emb, review_emb], axis=1)

    output_path = OUT_DIR / f"airbnb_sbert_embeddings_{split}.parquet"
    embeddings.to_parquet(output_path, index=False)
    print(f"Wrote {output_path}")

In [ ]:
for split in sorted(RUN_SPLITS):
    feature_keys = pd.read_parquet(
        OUT_DIR / f"airbnb_features_{split}.parquet",
        columns=["city", "snapshot", "listing_id"],
    )
    embedding_keys = pd.read_parquet(
        OUT_DIR / f"airbnb_sbert_embeddings_{split}.parquet",
        columns=["city", "snapshot", "listing_id"],
    )
    check = feature_keys.merge(
        embedding_keys,
        on=["city", "snapshot", "listing_id"],
        how="left",
        indicator=True,
    )
    print(split)
    print(check["_merge"].value_counts())